# View 4B: who produced the benchmark scores behind the capability index?

**Question:** For the models in View 4, how many benchmark scores were run by Epoch AI, how many come from a third party and how many come from a paper or model card published by a developer, by release year and by access group?

**Method in one paragraph.** Download the Epoch Benchmarking Hub ZIP. Stack the 58 benchmark files into one table of scores. Label each score by who produced it. Match the Hub model names to the 277 models of the capability file. Keep one score per model, benchmark and kind of source (the highest, if there are several). Count scores by release year and access group (the same `access_group` rule as View 4). Write one CSV that the app reads.

**Data licence:** Epoch AI Benchmarking Hub, Creative Commons Attribution 4.0. Credit: Epoch AI, 'Capabilities & benchmarking', epoch.ai/benchmarks. Benchmark questions and answers belong to their creators, and data from external projects keeps its own licence.

**Counting rule (fixed):** One score = one model, one benchmark and one kind of source. Versions of a model (for example high, low and max effort) are merged and the highest score is kept. A benchmark with two kinds of source for the same model counts twice. A benchmark with no result is not counted.

**Not checked by a second person yet:** the list of source labels (`manual/source_evidence_labels.csv`), the model name matches (`processed/hub_name_matches.csv`).

In [1]:
# 1. Settings
from pathlib import Path
import hashlib, io, json, re, shutil, zipfile
from datetime import datetime, timezone
import pandas as pd

HERE = Path.cwd()                      # run the notebook from its own folder
RAW = HERE / "raw"
PROCESSED = HERE / "processed"
MANUAL = HERE / "manual"
APP_DATA = HERE.parents[1] / "data"    # stories/2_gap_to_us_models/data
CAPABILITY = HERE.parent / "eci_scores" / "processed" / "model_capability.csv"

ZIP_URL = "https://epoch.ai/data/benchmark_data.zip"
ZIP_FILE = RAW / "benchmark_data.zip"
REFRESH = False        # True = download the ZIP again even if one is already in raw/
COPY_TO_APP = True     # True = copy the two output CSVs into stories/2_gap_to_us_models/data/
STRICT_LABELS = True   # True = stop if a source name is missing from manual/source_evidence_labels.csv
US = "United States of America"

for folder in (RAW, PROCESSED, MANUAL):
    folder.mkdir(exist_ok=True)

In [2]:
# 2. Get the ZIP. Reuse the file in raw/ unless REFRESH is True. Record how and when.
import requests

log_path = RAW / "retrieval_log.json"
log = json.loads(log_path.read_text()) if log_path.exists() else {}

if REFRESH or not ZIP_FILE.exists():
    response = requests.get(ZIP_URL, timeout=120, headers={"User-Agent": "36104-A3P2-student-project"})
    response.raise_for_status()
    ZIP_FILE.write_bytes(response.content)
    how, retrieved = "downloaded", datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
else:
    how = "reused"
    retrieved = log.get("benchmark_data", {}).get("retrieved_utc", "not recorded")

digest = hashlib.sha256(ZIP_FILE.read_bytes()).hexdigest()
log["benchmark_data"] = {
    "url": ZIP_URL, "file": ZIP_FILE.name, "how": how, "retrieved_utc": retrieved,
    "bytes": ZIP_FILE.stat().st_size, "sha256": digest,
}
log_path.write_text(json.dumps(log, indent=2))
print(json.dumps(log["benchmark_data"], indent=2))

{
  "url": "https://epoch.ai/data/benchmark_data.zip",
  "file": "benchmark_data.zip",
  "how": "reused",
  "retrieved_utc": "2026-11-16T04:17:00Z (date of the files inside the ZIP, download time not recorded)",
  "bytes": 2311553,
  "sha256": "8e7c44abefe2d11bd161bc1ab3684d41953c813c802c70d10f8d90acc82e36a2"
}


In [3]:
# 3. Read the benchmark list and every benchmark file inside the ZIP
archive = zipfile.ZipFile(ZIP_FILE)
names = set(archive.namelist())
metadata = pd.read_csv(archive.open("benchmark_metadata.csv"))
metadata = metadata.dropna(subset=["source_file", "score_column"]).reset_index(drop=True)
print(len(metadata), "benchmarks have a file and a score column")

frames, problems = [], []
for item in metadata.itertuples():
    if item.source_file not in names:
        problems.append((item.benchmark, "file missing in ZIP"))
        continue
    table = pd.read_csv(archive.open(item.source_file))
    if item.score_column not in table.columns:
        problems.append((item.benchmark, f"score column '{item.score_column}' missing"))
        continue
    score = pd.to_numeric(table[item.score_column], errors="coerce") * float(item.scale)
    is_external = item.source_file.endswith("_external.csv")   # Epoch's own runs have no "_external" in the file name
    frames.append(pd.DataFrame({
        "benchmark": item.benchmark,
        "hub_model_version": table["Model version"],
        "score": score,                       # share of the maximum, 0 to 1 (Hub scale)
        "is_external": is_external,
        "source_raw": table["Source"] if (is_external and "Source" in table.columns) else None,
    }))
rows = pd.concat(frames, ignore_index=True)
print("problems:", problems)
print(len(rows), "rows,", rows.benchmark.nunique(), "benchmarks,", int(rows.score.isna().sum()), "rows without a score,",
      int(rows.hub_model_version.isna().sum()), "rows without a model name")

58 benchmarks have a file and a score column
problems: []
5602 rows, 58 benchmarks, 8 rows without a score, 258 rows without a model name


In [4]:
# 4. Label every row by who produced the score
def clean_source(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    half = len(text) // 2
    return text[:half] if len(text) % 2 == 0 and text[:half] == text[half:] else text   # 'XX' typed twice -> 'X'

labels = pd.read_csv(MANUAL / "source_evidence_labels.csv")
label_of = dict(zip(labels.source_key, labels.evidence_type))
display_of = dict(zip(labels.source_key, labels.source))

def classify(row):
    if not row.is_external:
        return "epoch", "Epoch run"
    if pd.isna(row.source_raw):
        return "third_party", f"Source not stated ({row.benchmark})"
    key = clean_source(row.source_raw).casefold()
    if key not in label_of:
        return "unreviewed", clean_source(row.source_raw)
    return ("developer" if label_of[key] == "developer_report" else "third_party"), display_of[key]

kind_source = rows.apply(classify, axis=1, result_type="expand")
rows["kind"], rows["source"] = kind_source[0], kind_source[1]
missing = sorted(rows.loc[rows.kind == "unreviewed", "source"].unique())
print("sources not in the label table:", missing)
if STRICT_LABELS and missing:
    raise SystemExit("Add these sources to manual/source_evidence_labels.csv (developer_report or third_party), then run again.")
print(rows.kind.value_counts().to_string())

sources not in the label table: []
kind
third_party    2671
epoch          1680
developer      1251


In [5]:
# 5. Match Hub model names to the models in the capability file
capability = pd.read_csv(CAPABILITY)
EFFORT = r"_(high|low|max|medium|xhigh|minimal|none|unknown|auto|default|thinking)$"
STYLE = r"-(instruct|chat)$"
DATE = r"-(\d{4}-\d{2}-\d{2}|\d{8})$"

def match_key(name):
    text = str(name).lower()
    text = re.sub(EFFORT, "", text)
    text = re.sub(STYLE, "", text)
    text = re.sub(DATE, "", text)
    return re.sub(r"[^a-z0-9]", "", text)

by_key = {}
for model in capability.model:
    by_key.setdefault(match_key(model), []).append(model)

versions = rows.hub_model_version.dropna().unique()
match = {v: by_key[match_key(v)][0] for v in versions if len(by_key.get(match_key(v), [])) == 1}
rows["model"] = rows.hub_model_version.map(match)
pd.DataFrame({"hub_model_version": list(match), "capability_model": list(match.values())}) \
  .sort_values(["capability_model", "hub_model_version"]).to_csv(PROCESSED / "hub_name_matches.csv", index=False)
ambiguous = [v for v in versions if len(by_key.get(match_key(v), [])) > 1]
print(len(match), "of", len(versions), "Hub versions matched;", len(ambiguous), "ambiguous (left unmatched)")
print(rows.model.nunique(), "of", len(capability), "capability models have at least one Hub row")

492 of 937 Hub versions matched; 0 ambiguous (left unmatched)
191 of 277 capability models have at least one Hub row


In [6]:
# 6. Access group and release year, using the same rule as View 4 (src/context.py access_group)
open_weights = capability.epoch_accessibility_group.eq("Open weights")
group = pd.Series("excluded", index=capability.index)
group[capability.country.eq(US) & ~open_weights] = "restricted"
group[open_weights] = "accessible"
group[~open_weights & capability.country.notna() & ~capability.country.eq(US)] = "accessible"
capability["access_group"] = group
capability["release_year"] = capability.release_date.astype(str).str[:4]
print(capability.access_group.value_counts().to_string())

access_group
accessible    144
restricted    116
excluded       17


In [7]:
# 7. One score per model, benchmark and kind of source (highest kept)
scored = rows.dropna(subset=["model", "score"])
scored = scored.sort_values("score", ascending=False).drop_duplicates(["model", "benchmark", "kind"])
scored = scored.merge(capability[["model", "release_year", "access_group"]], on="model")
scored = scored[scored.access_group.isin(["accessible", "restricted"])]
out = scored[["model", "release_year", "access_group", "benchmark", "kind", "source", "score"]].copy()
out["score_pct"] = (out.score * 100).round(2)
out = out.drop(columns="score").sort_values(["release_year", "access_group", "model", "kind", "benchmark"])
out.to_csv(PROCESSED / "hub_scores_by_model.csv", index=False)

coverage = capability.groupby(["release_year", "access_group"]).model.nunique().rename("models_in_capability_file").reset_index()
with_scores = out.groupby(["release_year", "access_group"]).model.nunique().rename("models_with_hub_scores").reset_index()
coverage = coverage.merge(with_scores, how="left").fillna({"models_with_hub_scores": 0})
coverage.to_csv(PROCESSED / "hub_model_coverage.csv", index=False)
print(len(out), "scores,", out.model.nunique(), "models,", out.benchmark.nunique(), "benchmarks")
print(out.groupby(["release_year", "access_group", "kind"]).size().unstack(fill_value=0).to_string())
print(coverage.to_string(index=False))

2093 scores, 174 models, 57 benchmarks
kind                       developer  epoch  third_party
release_year access_group                               
2023         accessible          169      5           17
             restricted            8      2            3
2024         accessible           71     31           49
             restricted           19     41           67
2025         accessible            0     33           63
             restricted            4    144          243
2026         accessible            0    183          202
             restricted            1    331          407
release_year access_group  models_in_capability_file  models_with_hub_scores
        2023   accessible                         31                    23.0
        2023     excluded                         11                     0.0
        2023   restricted                          9                     2.0
        2024   accessible                         37                    21.0
      

In [8]:
# 8. Checks that could fail
assert out.duplicated(["model", "benchmark", "kind"]).sum() == 0, "a model has two scores for one benchmark and kind"
assert out.groupby("model").release_year.nunique().max() == 1, "a model sits in two years"
assert out.score_pct.between(0, 100).all(), "a score is outside 0 to 100"
assert set(out.kind) <= {"epoch", "third_party", "developer"}
print("All checks passed")

All checks passed


In [9]:
# 9. Data currency and copy to the app
currency = pd.DataFrame([{
    "dataset": "Epoch Benchmarking Hub benchmark_data.zip (hub_scores_by_model.csv)",
    "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "zip_retrieved_utc": log["benchmark_data"]["retrieved_utc"], "zip_how": log["benchmark_data"]["how"],
    "zip_sha256": log["benchmark_data"]["sha256"],
    "scores": len(out), "models": out.model.nunique(), "benchmarks": out.benchmark.nunique(),
    "note": "Release year and access group come from model_capability.csv (17 Oct 2026 snapshot).",
}])
currency.to_csv(PROCESSED / "hub_data_currency.csv", index=False)
if COPY_TO_APP:
    APP_DATA.mkdir(exist_ok=True)
    for name in ("hub_scores_by_model.csv", "hub_model_coverage.csv", "hub_data_currency.csv"):
        shutil.copy(PROCESSED / name, APP_DATA / name)
    print("Copied to stories/2_gap_to_us_models/data")

Copied to stories/2_gap_to_us_models/data
